# The DAG and Query Execution Plans: Parsed, Analyzed, Optimized, and Physical

**Module**: 04_execution_engine  
**Notebook**: 02_dag_and_execution_plans.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Catalyst converts raw code into the 4 plan stages: Parsed Logical Plan, Analyzed Logical Plan, Optimized Logical Plan, and Physical Plan.
- **Why does this topic exist?**: Reading `df.explain()` is the #1 forensic skill for Senior Data Engineers to diagnose query performance.
- **What problem does it solve?**: Queries fail or run slowly because of unnoticed Cartesian joins, missing predicate pushdown, or unnecessary shuffles.
- **Where is it used in real-world Data Engineering?**: Optimizing multi-hour enterprise ETL jobs by analyzing explain plan trees.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Lazy evaluation, Catalyst basics |
| **Required depth** | Deep working knowledge: can read logical and physical plans |
| **Data Engineer** | Read `explain()` output and identify scans, filters, exchanges, joins, and aggregates. |
| **Senior Data Engineer** | Detect plan regressions and choose physical strategies. |
| **Spark specialist** | Explain Catalyst phases, cost-based decisions, and AQE re-planning. |
| **Next topic** | `04_execution_engine/03_dag_scheduler.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A query plan is the contract between your code and the execution engine. The logical plan says what to
compute; the physical plan says how. Two queries that look equivalent can produce different plans because of
statistics, join order, or a hidden cast. Reading plans is the difference between guessing and tuning.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Parsed plan** | Unresolved syntax tree | Names/types not yet resolved |
| **Analyzed plan** | Resolved against catalog | Columns and functions validated |
| **Optimized plan** | Rule-based rewrites | Pushdown, pruning, constant folding |
| **Physical plan** | Executable operators | Join strategy, exchange, scan |
| **DAG** | Stage graph | Split at shuffle boundaries |
| **AQE** | Runtime re-optimization | Coalesce partitions, skew handling |

### How it works, step by step

1. The parser turns SQL/DataFrame operations into a logical plan.
2. The analyzer resolves names and types using the catalog.
3. Optimizer rules rewrite the plan.
4. The physical planner selects operators and strategies.
5. The DAGScheduler converts the physical plan into stages.
6. AQE may adjust the plan between stages using runtime statistics.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. The 4 Stages of Catalyst Query Planning

```
User Code / SQL
       |
       v
1. PARSED LOGICAL PLAN (Unresolved: verifies syntax; column names unresolved)
       |
       v (Catalog Resolution: checks table schemas, column types)
2. ANALYZED LOGICAL PLAN (Resolved: all columns and functions verified)
       |
       v (Rule-Based Optimization: Predicate Pushdown, Column Pruning, Constant Folding)
3. OPTIMIZED LOGICAL PLAN (Minimal algebraic operations)
       |
       v (Cost-Based Model: Join Strategy Selection, Algorithm Choice)
4. PHYSICAL PLAN (Concrete executable operators: FileScan, HashAggregate, SortMergeJoin)
       |
       v (Tungsten: WholeStageCodegen)
  Java Bytecode Execution
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Execution_Plans_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: Inspecting All 4 Plan Stages with `explain(True)`

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
products = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/products.csv")

query_df = (
    orders
    .filter("quantity > 2")
    .join(products, on="product_id")
    .filter("category = 'Electronics'")
    .select("order_id", "product_name", "quantity", "unit_price")
)

# explain(True) prints Parsed, Analyzed, Optimized, and Physical plans
query_df.explain(True)

## 4. Modern Formatted Execution Plan with `explain('formatted')`

In [ ]:
print("=== Formatted Plan Output ===")
query_df.explain("formatted")

---

## Experiment: Predicate Pushdown Observation in Physical Plan

- **1. What are we testing?**: Verify in the Physical Plan that'category = Electronics'is pushed down directly into the FileScan node.
- **2. What do we expect?**: PushedFilters in FileScan contains StringEqualTo(category, Electronics).

In [ ]:
print("Check FileScan PushedFilters:")
query_df.explain("cost")

**3. What happened?**: FileScan showed PushedFilters directly reading matching rows.  
**4. Why did it happen?**: Catalyst moved the filter condition past the join directly into the table reader.  
**5. Practical Engineering Lesson**: Always verify that your filters appear under PushedFilters in physical plans to prevent scanning unneeded data.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Plan review before a production release

- **Industry**: Banking
- **Scenario**: A team reviews a changed reporting query before it reaches production.
- **Problem**: A small code change can introduce a cross join or a full scan that misses the SLA.
- **Why this concept is relevant**: Comparing the physical plans before and after the change catches expensive operators early.
- **Implementation idea**: Store `explain('formatted')` output in code review, diff the plans, and run on a representative sample.
- **Expected result**: A reviewed plan with no unexpected exchanges or nested loops.
- **Alternative**: A warehouse query-plan review.
- **Trade-offs**: Spark plans expose distributed operators; warehouse plans expose storage/statistics details.

### Use Case 2: Diagnosing a slow join

- **Industry**: E-commerce
- **Scenario**: A join that used to be fast now shuffles both sides.
- **Problem**: Table statistics changed, or a hint was removed, so the optimizer no longer broadcasts the small side.
- **Why this concept is relevant**: The physical plan shows the chosen join strategy and the exchange nodes.
- **Implementation idea**: Inspect `explain()`, check table sizes/statistics, and apply a broadcast hint only if the small side is truly small.
- **Expected result**: A deliberate join strategy with measured improvement.
- **Alternative**: A cost-based optimizer in a warehouse.
- **Trade-offs**: Spark's CBO/AQE is capable but depends on accurate statistics.

---

## Business Value

- Catches expensive plan changes before they affect SLAs.
- Makes performance discussions evidence-based.
- Helps teams choose between SQL and DataFrame formulations.
- Supports safer migrations and capacity planning.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Use `explain('formatted')` and the SQL tab together.
- Collect table statistics for CBO; stale stats mislead the optimizer.
- Do not force hints without verifying sizes.
- Watch for implicit casts that prevent partition pruning.
- Treat plan changes as code changes in review.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Read a formatted plan

**Steps**

1. Build a join + aggregation query.
2. Run `explain('formatted')`.
3. Identify the scan, filter, exchange, join, and aggregate nodes.
4. Compare with the SQL tab after execution.

**What to observe**: The formatted plan groups operators and shows physical details.

**Healthy result**: You can explain each operator and where data moves.

**Problematic result**: An unexpected `Exchange` or `CartesianProduct` signals a design issue.

**Engineering takeaway**: Read plans from the bottom up: sources, filters, joins, exchanges, output.

---

## Trade-offs and Alternatives

### Advantages

- Transparent optimization
- Rich operator details
- AQE adapts at runtime

### Limitations

- Plans are verbose
- Statistics can be stale
- AQE can make plans harder to compare

### When to use it

- Performance tuning
- Code review
- Join debugging
- Migration validation

### When not to use it

- Optimizing without a plan
- Assuming the plan is stable across data sizes
- Reading only the logical plan

### Alternatives inside Spark

- `explain()`
- SQL tab
- AQE

### Alternatives outside Spark

- Warehouse query profilers
- Database `EXPLAIN`

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify WholeStageCodegen in Plan
**Problem Statement**: Run explain() and identify which physical plan operators are prefixed with an asterisk (*), indicating WholeStageCodegen.  
**Input / Context**: query_df

In [ ]:
# Exercise 1: Identify codegen operators

### Exercise 2 (Intermediate): Detect Exchange (Shuffle) Operators
**Problem Statement**: Write a query that groups and sorts data, and count how many Exchange nodes appear in explain().  
**Input / Context**: orders DataFrame

In [ ]:
# Exercise 2: Count Exchange nodes

### Exercise 3 (Advanced): Analyze Join Strategy in Physical Plan
**Problem Statement**: Inspect query_df and determine whether Spark selected BroadcastHashJoin or SortMergeJoin.  
**Input / Context**: query_df

In [ ]:
# Exercise 3: Join strategy identification

---

## Production Debugging Exercise

### The plan that changed without code

**Scenario**: A query becomes slower after a data reload even though the SQL is unchanged.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Compare the physical plans before and after.
2. Check table statistics and file sizes.
3. Look for a join strategy change.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify WholeStageCodegen in Plan):
```python
print("Operators with * run inside WholeStageCodegen:")
for line in query_df._jdf.queryExecution().executedPlan().toString().split("\n"):
    if"*"in line:
        print("->", line.strip())
```

#### Solution 2 (Detect Exchange (Shuffle) Operators):
```python
p = orders.groupBy("status").count().orderBy("count")
plan_str = p._jdf.queryExecution().executedPlan().toString()
print("Exchange count:", plan_str.count("Exchange"))
```

#### Solution 3 (Analyze Join Strategy in Physical Plan):
```python
plan = query_df._jdf.queryExecution().executedPlan().toString()
if"BroadcastHashJoin"in plan:
    print("Selected Join: BroadcastHashJoinExec")
elif"SortMergeJoin"in plan:
    print("Selected Join: SortMergeJoinExec")
else:
    print("Other Join Strategy Selected")
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **4 Plan Levels**: Parsed -> Analyzed -> Optimized -> Physical.
- **explain(True)**: The universal tool for inspecting Catalyst decision-making.
- **PushedFilters**: Indicates storage-level filtering (predicate pushdown).

### Essential Technical Terms
- `Parsed Plan`: Unresolved syntax tree.
- `Analyzed Plan`: Resolved against catalog metadata.
- `Physical Plan`: Executable algorithm graph.

### Common Engineering Mistakes
- **Ignoring Exchange nodes**: Missing that your query is doing 5 unnecessary network shuffles.
- **Assuming queries execute imperatively**: Catalyst reorders filters and joins for optimal execution.

### Interview Questions on This Topic
- **Q: What is the difference between Logical Plan and Physical Plan?**
  - *A*: The Logical Plan describes WHAT data to compute; the Physical Plan describes HOW to compute it (which concrete algorithms and join strategies to execute).
- **Q: What does an asterisk (*) next to an operator in explain() mean?**
  - *A*: It signifies that the operator participates in Whole-Stage Code Generation (Tungsten).


### More Interview Questions

- **Q: What are the main phases of Catalyst?**
  - *A*: Parsing, analysis, logical optimization, physical planning, and execution (with AQE possibly re-optimizing at runtime).
- **Q: What is the difference between a logical and a physical plan?**
  - *A*: The logical plan describes what to compute; the physical plan describes the executable operators and data movement.

### What to Study Next
Open 04_execution_engine/03_dag_scheduler.ipynb to dissect the DAGScheduler.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')